In [ ]:
import pandas as pd
import numpy as np

xml_path = "dades/Xml_20261119_124136_1.xml"

# Preprocessing

In this file we do the preprocessing of the data file `Xml_20261119_124136_1.xml`, which we obtained from a Government of Spain database. As this .xml file has the location, cause, time, and burned area stored differently, we are going to preprocess them independently, with the objective of merging them at the end.

## Location

In [ ]:
location_columns = ["numeroparte", "idcomunidad", "idprovincia",
                    "idmunicipio", "latitud", "longitud"]

locations = pd.read_xml(xml_path, xpath="./Pif/pif_localizacion")[location_columns]
locations["codi_municipi"] = (
    locations["idprovincia"].astype("Int64").astype("string").str.zfill(2)
    + locations["idmunicipio"].astype("Int64").astype("string").str.zfill(3)
)
locations["codi_municipi"] = locations["codi_municipi"].mask(locations["idmunicipio"]==0)
locations = locations.drop(["idcomunidad", "idmunicipio"], axis=1)
locations["coords_imputades"] = (locations["latitud"].isna() & locations["codi_municipi"].notna())
locations.info()
locations.loc[locations["codi_municipi"].notna()].head()

In [ ]:
provincies = ["08", "17", "25", "43"]
columnes = ["COD_INE", "COD_PROV",
            "NOMBRE_ACTUAL",
            "PROVINCIA",
            "LONGITUD_ETRS89_REGCAN95",
            "LATITUD_ETRS89_REGCAN95"]

path = "dades/MUNICIPIOS.csv"

municipis_info = pd.read_csv(
    path,
    encoding="cp1252",
    sep=";",
    dtype="string",
)

municipis_info = municipis_info.loc[municipis_info["COD_PROV"].isin(provincies), columnes]
municipis_info["codi_municipi"] = municipis_info["COD_INE"].str[:5]
municipis_info = municipis_info.drop(["COD_INE", "COD_PROV"], axis=1)
municipis_info.info()
municipis_info.head()

<class 'pandas.core.frame.DataFrame'>
Index: 947 entries, 862 to 6544
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   NOMBRE_ACTUAL             947 non-null    string
 1   PROVINCIA                 947 non-null    string
 2   LONGITUD_ETRS89_REGCAN95  947 non-null    string
 3   LATITUD_ETRS89_REGCAN95   947 non-null    string
 4   codi_municipi             947 non-null    string
dtypes: string(5)
memory usage: 44.4 KB


,NOMBRE_ACTUAL,PROVINCIA,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,codi_municipi
862,Abrera,Barcelona,"1,901568685","41,5163976",08001
863,Aguilar de Segarra,Barcelona,"1,631134116","41,73926837",08002
864,Alella,Barcelona,"2,295316458","41,49361157",08003
865,Alpens,Barcelona,"2,101098216","42,11928971",08004
866,L'Ametlla del Vallès,Barcelona,"2,261438098","41,66955386",08005


In [ ]:
comarques = pd.read_csv(
    "dades/corrMpisComsAtps.csv",
    sep=";",
    skiprows=3,
    usecols=["Codi municipi", "Nom comarca", "Nom municipi"],
    dtype={"Codi municipi": "string"},
)#[["Codi municipi", "Nom comarca"]].copy()

comarques["codi_municipi"] = (
    comarques["Codi municipi"]
    .astype("string").str.strip()
    .str[:5]
    .str.zfill(5)
)
comarques = comarques[["codi_municipi", "Nom comarca"]]
comarques.info()
comarques.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947 entries, 0 to 946
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   codi_municipi  947 non-null    string
 1   Nom comarca    947 non-null    object
dtypes: object(1), string(1)
memory usage: 14.9+ KB


,codi_municipi,Nom comarca
0,25001,Pallars Jussà
1,08001,Baix Llobregat
2,25002,Noguera
3,25003,Urgell
4,08002,Bages


In [ ]:
merged1 = locations.merge(
    municipis_info, 
    how="left", 
    on="codi_municipi")
merged1["latitud"] = merged1["latitud"].fillna(merged1["LATITUD_ETRS89_REGCAN95"])
merged1["longitud"] = merged1["longitud"].fillna(merged1["LONGITUD_ETRS89_REGCAN95"])
merged1 = merged1.drop(["LONGITUD_ETRS89_REGCAN95", "LATITUD_ETRS89_REGCAN95"], axis=1)
df_loc = merged1.merge(
    comarques,
    how="left",
    on="codi_municipi"
)
df_loc = df_loc.rename(columns={
    "NOMBRE_ACTUAL": "municipi",
    "PROVINCIA": "provincia",
    "Nom comarca": "comarca" 
})
df_loc = df_loc.iloc[:, [0,3,5,7,6,1,2,4]]
df_loc.loc[merged1["codi_municipi"].notna()].head()


,numeroparte,longitud,coords_imputades,provincia,municipi,idprovincia,latitud,codi_municipi
6436,1983082674,"2,718588087",True,Barcelona,Tordera,8,"41,70171978",08284
6437,1983082675,"2,258205146",True,Barcelona,Santa Maria de Besora,8,"42,12676151",08253
6438,1983082676,"2,187220461",True,Barcelona,Montcada i Reixac,8,"41,48002694",08125
6439,1983082677,"2,24354259",True,Barcelona,Malla,8,"41,89460649",08111
6440,1983082678,"2,67140181",True,Barcelona,Fogars de la Selva,8,"41,73512796",08082


In [ ]:
df_loc.loc[(df_loc["municipi"] == "Tordera") & (df_loc["coords_imputades"] == False), ["municipi", "latitud", "longitud"]]

,municipi,latitud,longitud
16161,Tordera,41.689955,2.688771
16166,Tordera,41.704561,2.771634
16225,Tordera,41.657408,2.646891
17826,Tordera,41.694518,2.711582
17839,Tordera,41.723349,2.71506
...,...,...,...
27853,Tordera,41.682202,2.713944
27883,Tordera,41.712664,2.721118
27909,Tordera,41.657089,2.642665
29355,Tordera,41.718887,2.736154


## Causes

In [ ]:
causes = pd.read_xml(xml_path, xpath="./Pif/pif_causa")
causes.info()
causes.head()

In [ ]:
columnes_causes = ["numeroparte", "idcausa", "causa"]

causes_egif_ca = pd.read_csv("dades/causes_egif_ca.csv")
df_causes = causes.merge(
    causes_egif_ca,
    on="idcausa",
    how="left",
    validate="many_to_one",
)
df_causes["causa"] = df_causes["causa"].mask(df_causes["causa"] == "Desconeguda")
df_causes = df_causes[columnes_causes]
df_causes.head()


,numeroparte,idcausa,causa
0,1968080028,220,Altres cremes ramaderes
1,1968080029,500,NaN
2,1968080030,310,Outras causas por ferrocarril
3,1968080031,500,NaN
4,1968080032,240,Altres tipus de fogueres (conegudes)


## Temps

In [ ]:
temps_columnes = ["numeroparte", "deteccion"]
df_temps = pd.read_xml(xml_path, xpath="./Pif/pif_tiempos")[temps_columnes]
df_temps.head()

In [ ]:
df_temps["deteccion"] = df_temps["deteccion"].astype("string").str.strip().str[:10]
df_temps["deteccion"] = pd.to_datetime(df_temps["deteccion"], errors="coerce", format="%Y-%m-%d")
df_temps = df_temps.rename(columns={
    "deteccion": "data"
})
df_temps.head()

,numeroparte,data
0,1968080028,1968-01-10
1,1968080029,1968-01-10
2,1968080030,1968-01-10
3,1968080031,1968-01-11
4,1968080032,1968-01-11


## Hectares

Total recorded burned area includes wooded forest, non-wooded forest, agricultural, and other non-forest land, in hectares. Missing components are skipped, and totals with no recorded area remain missing.

In [ ]:
hectares_columnes = [
    "superficiearboladatotal",
    "superficienoarboladatotal",
    "superficienoarboladaagricola",
    "superficienoarboladaotras",
]

hectares = pd.read_xml(xml_path, xpath="./Pif/pif_perdidas")
hectares = hectares.reindex(columns=["numeroparte"] + hectares_columnes)
hectares["total_hectares"] = hectares[hectares_columnes].sum(axis=1, min_count=1)
df_hectares = hectares[["numeroparte", "total_hectares"]]
df_hectares.info()
df_hectares.head()

## Merge and export csv file

In [ ]:
merged1 = df_loc.merge(
    df_causes,
    how="inner",
    on="numeroparte")
df = merged1.merge(
    df_temps,
    how="inner",
    on="numeroparte"
)

df = df.merge(
    df_hectares,
    how="left",
    on="numeroparte",
    validate="one_to_one",
)

df.to_csv("Clean data/CleanGovEsp.csv")
df.head()